# MSCS Assessment and Accountability Analyst Hiring Exercise

**Scenario:** A senior leader is preparing for a coaching meeting with three school principals focused
on improving student attendance outcomes. The senior leader has requested attendance and
absenteeism data from our office to inform this conversation. As the analyst, your role is to prepare
both a data file and a written analysis that the senior leader can use to guide a productive, data-
informed discussion with the principals.

## Import packages

Import the packages necessary for cleaning and merging the three datasets.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## Set Up the File Pathways

Dynamically find the file path for easy importing of datasets.

In [2]:
# get current notebook directory
base_dir = Path.cwd()
print(base_dir)

# go up one level
project_root = base_dir.parent
# print(project_root)

# build path to csv folder
data_dir = project_root / 'data' / 'csv'

# print(data_dir)

/Users/ajgafford/Documents/Projects/Python/MSCS/notebooks


## Import the Datasets

There are three CSV files:

- `Student_Demographic.csv`
- `Student_Attendance.csv`
- `Student_Enrollment.csv`

All three were created by exporting the raw Excel files into CSVs.

In [3]:
demographic_file = data_dir / 'Student_Demographic.csv'
attendance_file = data_dir / 'Student_Attendance.csv'
enrollment_file = data_dir / 'Student_Enrollment.csv'

In [4]:
student_demographic = pd.read_csv(demographic_file)
student_attendance = pd.read_csv(attendance_file)
student_enrollment = pd.read_csv(enrollment_file)

## Check the Structure

Get a quick glance of the structure of each DataFrame.

In [5]:
student_demographic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Student*                 1301 non-null   object
 1   Student ID               1301 non-null   int64 
 2   Grade                    1301 non-null   object
 3   Student with Disability  1301 non-null   object
 4   SMS Medical Alert        461 non-null    object
dtypes: int64(1), object(4)
memory usage: 50.9+ KB


In [6]:
student_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   StudentID*       1301 non-null   int64  
 1   Attendance Rate  1301 non-null   float64
 2   Days Absent      1301 non-null   int64  
 3   Days Excused     1301 non-null   int64  
 4   Days Unexcused   1301 non-null   int64  
 5   Days Suspended   1301 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 61.1 KB


In [7]:
student_enrollment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   School         1301 non-null   object
 1   Student*       1301 non-null   object
 2   StudentID*     1301 non-null   int64 
 3   Days Enrolled  1301 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 40.8+ KB


**Takeaways:** There do not appear to be null values except for `SMS Medical Alert`. Would guess null in that field means no medical alert, but further exploration will reveal more. There are inconsistencies in field names that need fixing.

Which three schools represented?

In [8]:
student_enrollment['School'].unique()

array(['Littlebrook Elementary', 'Stevenson High', 'North Star Middle'],
      dtype=object)

Let's check the five number summary to see if there could potentially be in data anomolies.

In [9]:
student_attendance.describe()

,StudentID*,Attendance Rate,Days Absent,Days Excused,Days Unexcused,Days Suspended
count,1301.000000,1301.000000,1301.000000,1301.000000,1301.000000,1301.000000
mean,1617.148347,90.292083,15.068409,4.777863,8.289777,2.000769
std,930.881056,8.968996,13.678307,7.230534,9.108903,5.830094
min,4.000000,20.700000,0.000000,0.000000,0.000000,0.000000
25%,789.000000,87.000000,5.000000,0.000000,2.000000,0.000000
50%,1659.000000,92.700000,11.000000,2.000000,5.000000,0.000000
75%,2450.000000,96.600000,20.000000,6.000000,11.000000,1.000000
max,3155.000000,100.000000,142.000000,52.000000,67.000000,101.000000


There is potentially an outlier student with 142 days of absences, which is maybe the same student who has 101 days suspended.

In [10]:
student_enrollment['Days Enrolled'].describe()

count    1301.000000
mean      162.269792
std        35.599312
min        22.000000
25%       172.000000
50%       179.000000
75%       179.000000
max       179.000000
Name: Days Enrolled, dtype: float64

There are potential outliers on the low end of enrollment. 75% of students have been enrolled at least 172 days.

What are the unique medical conditions accounted for?

In [11]:
student_demographic['SMS Medical Alert'].unique()

array([nan, 'Chronic illness', 'Asthma'], dtype=object)

Looks like when the value is null it's just no medical alert.

## Data Cleaning

Rename any fields and address any inconsistencies in the data.

In [12]:
def clean_columns(df):
    df.columns = (
        df.columns
            .str.strip()
            .str.replace("*", "", regex=False)
            .str.lower()
            .str.replace(" ", "_")
    )

    return df

In [13]:
student_demographic = clean_columns(student_demographic)
student_demographic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   student                  1301 non-null   object
 1   student_id               1301 non-null   int64 
 2   grade                    1301 non-null   object
 3   student_with_disability  1301 non-null   object
 4   sms_medical_alert        461 non-null    object
dtypes: int64(1), object(4)
memory usage: 50.9+ KB


In [14]:
student_attendance = clean_columns(student_attendance)
student_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   studentid        1301 non-null   int64  
 1   attendance_rate  1301 non-null   float64
 2   days_absent      1301 non-null   int64  
 3   days_excused     1301 non-null   int64  
 4   days_unexcused   1301 non-null   int64  
 5   days_suspended   1301 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 61.1 KB


In [15]:
student_attendance = student_attendance.rename(columns={'studentid':'student_id'})
student_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       1301 non-null   int64  
 1   attendance_rate  1301 non-null   float64
 2   days_absent      1301 non-null   int64  
 3   days_excused     1301 non-null   int64  
 4   days_unexcused   1301 non-null   int64  
 5   days_suspended   1301 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 61.1 KB


In [16]:
student_enrollment = clean_columns(student_enrollment)
student_enrollment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   school         1301 non-null   object
 1   student        1301 non-null   object
 2   studentid      1301 non-null   int64 
 3   days_enrolled  1301 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 40.8+ KB


In [17]:
student_enrollment = student_enrollment.rename(columns={'studentid':'student_id'})
student_enrollment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   school         1301 non-null   object
 1   student        1301 non-null   object
 2   student_id     1301 non-null   int64 
 3   days_enrolled  1301 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 40.8+ KB


Drop any duplicate rows.

In [18]:
# should return 0 if no duplicates
print(f"Demographic Duplicates: {student_demographic.duplicated(subset=['student_id']).sum()}")
print(f"Attendance Duplicates: {student_attendance.duplicated(subset=['student_id']).sum()}")
print(f"Enrollment Duplicates: {student_enrollment.duplicated(subset=['student_id']).sum()}")

Demographic Duplicates: 1
Attendance Duplicates: 1
Enrollment Duplicates: 1


1 student is duplicated in all three datasets.

In [19]:
# show all duplicates based on student_id
duplicates_demo = student_demographic[student_demographic.duplicated(subset=['student_id'], keep=False)]
print("Demographic Duplicates:")
print(duplicates_demo)

duplicates_att = student_attendance[student_attendance.duplicated(subset=['student_id'], keep=False)]
print("\nAttendance Duplicates:")
print(duplicates_att)

duplicates_enroll = student_enrollment[student_enrollment.duplicated(subset=['student_id'], keep=False)]
print("\nEnrollment Duplicates:")
print(duplicates_enroll)

Demographic Duplicates:
     student  student_id grade student_with_disability sms_medical_alert
500  J. GRAY        1282     9                      No               NaN
501  J. GRAY        1282     9                      No               NaN

Attendance Duplicates:
      student_id  attendance_rate  days_absent  days_excused  days_unexcused  \
1161        1282             97.8            4             1               3   
1162        1282             93.9           11             5               6   

      days_suspended  
1161               0  
1162               0  

Enrollment Duplicates:
              school  student  student_id  days_enrolled
1185  Stevenson High  J. GRAY        1282            179
1186  Stevenson High  J. GRAY        1282            179


**Takeaways:** In the Demographic and Enrollment datasets, J. Gray (1282) is double represented, so can safely drop those two rows. However, in the Attendance dataset, this student has two unique records.

In [20]:
# drop duplicate, keep first
student_demographic = student_demographic.drop_duplicates(subset=['student_id'], keep='first')
student_enrollment = student_enrollment.drop_duplicates(subset=['student_id'], keep='first')

In [21]:
# should return 0 if no duplicates
print(f"Demographic Duplicates: {student_demographic.duplicated(subset=['student_id']).sum()}")
print(f"Enrollment Duplicates: {student_enrollment.duplicated(subset=['student_id']).sum()}")

Demographic Duplicates: 0
Enrollment Duplicates: 0


For the duplicated attendance record, my solution for the time being is to assume that this is the same student and merge the two records into one. Essentially, combine the days fields through taking the sum. I will recalculate `attendance_rate` once I have merged the three datasets.

In [22]:
# isolate the columns to sum
attendance_cols = ['days_absent', 'days_excused', 'days_unexcused', 'days_suspended']

# sum duplicate student rows
student_attendance = (
    student_attendance.groupby('student_id')[attendance_cols]
    .sum()
    .reset_index()
)

In [23]:
duplicates_att = student_attendance[student_attendance.duplicated(subset=['student_id'], keep=False)]
print("\nAttendance Duplicates:")
print(duplicates_att)


Attendance Duplicates:
Empty DataFrame
Columns: [student_id, days_absent, days_excused, days_unexcused, days_suspended]
Index: []


In [24]:
print(f"Attendance Duplicates: {student_attendance.duplicated(subset=['student_id']).sum()}")

Attendance Duplicates: 0


In [25]:
print(student_attendance.loc[student_attendance['student_id'] == 1282])

     student_id  days_absent  days_excused  days_unexcused  days_suspended
500        1282           15             6               9               0


In [26]:
student_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   student_id      1300 non-null   int64
 1   days_absent     1300 non-null   int64
 2   days_excused    1300 non-null   int64
 3   days_unexcused  1300 non-null   int64
 4   days_suspended  1300 non-null   int64
dtypes: int64(5)
memory usage: 50.9 KB


In [27]:
student_demographic.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1300 entries, 0 to 1300
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   student                  1300 non-null   object
 1   student_id               1300 non-null   int64 
 2   grade                    1300 non-null   object
 3   student_with_disability  1300 non-null   object
 4   sms_medical_alert        461 non-null    object
dtypes: int64(1), object(4)
memory usage: 60.9+ KB


In [28]:
student_enrollment.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1300 entries, 0 to 1300
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   school         1300 non-null   object
 1   student        1300 non-null   object
 2   student_id     1300 non-null   int64 
 3   days_enrolled  1300 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 50.8+ KB


Let's inspect the potential outlier(s) in terms of `days_absent`.

In [29]:
print(f"Records with days absent >= 100: {len(student_attendance[student_attendance['days_absent'] >= 100])}")

Records with days absent >= 100: 1


In [30]:
print(student_attendance[student_attendance['days_absent'] >= 100])

     student_id  days_absent  days_excused  days_unexcused  days_suspended
668        1703          142             7              34             101


While this student is an outlier, I should keep it in the dataset as it likely just represents a student with a lengthy out of school suspension. It should not be disregarded, but it can be considered in future analysis.

## Data Merging

Now that the data is clean, I can confidently merge the three datasets.

Each dataset has a unique primary key (`student_id`) which will serve as the best way to merge the datasets into one.

In [31]:
# merge attendance with demographics
students = student_attendance.merge(
    student_demographic,
    on='student_id',
    how='left'
)

# merge in enrollment data
students = students.merge(
    student_enrollment[['student_id', 'school', 'days_enrolled']],
    on='student_id',
    how='left'
)

# reorder columns for readability
students = students[['student_id', 'student', 'school', 'grade', 'student_with_disability', 'sms_medical_alert',
                       'days_enrolled', 'days_absent', 'days_excused', 'days_unexcused', 'days_suspended']].copy()

# quick check
print(students.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1300 entries, 0 to 1299
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   student_id               1300 non-null   int64 
 1   student                  1300 non-null   object
 2   school                   1300 non-null   object
 3   grade                    1300 non-null   object
 4   student_with_disability  1300 non-null   object
 5   sms_medical_alert        461 non-null    object
 6   days_enrolled            1300 non-null   int64 
 7   days_absent              1300 non-null   int64 
 8   days_excused             1300 non-null   int64 
 9   days_unexcused           1300 non-null   int64 
 10  days_suspended           1300 non-null   int64 
dtypes: int64(6), object(5)
memory usage: 111.8+ KB
None


The merger was successful and the data is organized neatly.

---

## Feature Engineering

Additional features can be added to enhance analysis.

I can add back in the `attendance_rate` field. `attendance_rate` should be defined as:

$$ \text{attendance rate} = \frac{\text{days enrolled} - \text{days absent}}{\text{days enrolled}}$$

In [32]:
students['attendance_rate'] = (students['days_enrolled'] - students['days_absent']) / students['days_enrolled']
print(students.head(10))

   student_id                 student                  school grade  \
0           4               L. MACLIN       North Star Middle     8   
1           7             T. SHOCKLEY          Stevenson High     9   
2           8              W. BONILLA       North Star Middle     7   
3           9               T. MCGREW          Stevenson High    10   
4          10                T. RUFUS          Stevenson High    12   
5          13              J. SANCHEZ       North Star Middle     7   
6          14  T. FIGUEREDO CASTANEDA       North Star Middle     6   
7          17                Z. BROWN       North Star Middle     8   
8          23             E. HOLLOWAY  Littlebrook Elementary     2   
9          24             N. PHILMORE          Stevenson High    10   

  student_with_disability sms_medical_alert  days_enrolled  days_absent  \
0                      No               NaN            179            4   
1                      No               NaN            110          

I want to define a categorical field `attendance_status` to track if this student is currently chronically absent, at risk, or satisfactory. Based on the assignment definitions:

- **Chronic**: `attendance_rate`<= 90%
- **At Risk**: `attendance_rate`<= 95%
- **Satisfactory**: `attendance_rate` > 95%

In [33]:
def determine_attendance_status(row):
    if row['attendance_rate'] <= 0.9:
        return 'Chronic'
    elif row['attendance_rate'] <= 0.95:
        return 'At-Risk'
    else:
        return 'Satisfactory'

In [34]:
students['attendance_status'] = students.apply(determine_attendance_status, axis=1)
print(students.head(10))

   student_id                 student                  school grade  \
0           4               L. MACLIN       North Star Middle     8   
1           7             T. SHOCKLEY          Stevenson High     9   
2           8              W. BONILLA       North Star Middle     7   
3           9               T. MCGREW          Stevenson High    10   
4          10                T. RUFUS          Stevenson High    12   
5          13              J. SANCHEZ       North Star Middle     7   
6          14  T. FIGUEREDO CASTANEDA       North Star Middle     6   
7          17                Z. BROWN       North Star Middle     8   
8          23             E. HOLLOWAY  Littlebrook Elementary     2   
9          24             N. PHILMORE          Stevenson High    10   

  student_with_disability sms_medical_alert  days_enrolled  days_absent  \
0                      No               NaN            179            4   
1                      No               NaN            110          

While `absence_rate` would be implied through `attendance_rate`, it would be nice to have that directly for reporting.

In [35]:
students['absence_rate'] = 1 - students['attendance_rate']

# reorder columns for readability
students = students[['student_id', 'student', 'school', 'grade', 'student_with_disability', 'sms_medical_alert',
                       'days_enrolled', 'days_absent', 'days_excused', 'days_unexcused', 'days_suspended', 'attendance_rate',
                        'absence_rate', 'attendance_status']].copy()

print(students.head(10))

   student_id                 student                  school grade  \
0           4               L. MACLIN       North Star Middle     8   
1           7             T. SHOCKLEY          Stevenson High     9   
2           8              W. BONILLA       North Star Middle     7   
3           9               T. MCGREW          Stevenson High    10   
4          10                T. RUFUS          Stevenson High    12   
5          13              J. SANCHEZ       North Star Middle     7   
6          14  T. FIGUEREDO CASTANEDA       North Star Middle     6   
7          17                Z. BROWN       North Star Middle     8   
8          23             E. HOLLOWAY  Littlebrook Elementary     2   
9          24             N. PHILMORE          Stevenson High    10   

  student_with_disability sms_medical_alert  days_enrolled  days_absent  \
0                      No               NaN            179            4   
1                      No               NaN            110          

I could see knowing the percentage of absenses that are unexcused being important for leadership to know. Is a student missing because of truancy, rather than illness?

In [36]:
students['pct_unexcused_absence'] = np.where(
    students['days_absent'] == 0,
    0,
    students['days_unexcused'] / students['days_absent']
)
print(students.head(10))

   student_id                 student                  school grade  \
0           4               L. MACLIN       North Star Middle     8   
1           7             T. SHOCKLEY          Stevenson High     9   
2           8              W. BONILLA       North Star Middle     7   
3           9               T. MCGREW          Stevenson High    10   
4          10                T. RUFUS          Stevenson High    12   
5          13              J. SANCHEZ       North Star Middle     7   
6          14  T. FIGUEREDO CASTANEDA       North Star Middle     6   
7          17                Z. BROWN       North Star Middle     8   
8          23             E. HOLLOWAY  Littlebrook Elementary     2   
9          24             N. PHILMORE          Stevenson High    10   

  student_with_disability sms_medical_alert  days_enrolled  days_absent  \
0                      No               NaN            179            4   
1                      No               NaN            110          

I believe this is all I need at the student level.

---

## Create School Level Dataset

With the student level data, I can now focus on making school level data for one of my deliverables.

In [37]:
school_summary = students.groupby('school').agg(
    total_students=('student_id', 'count'),
    total_days_enrolled=('days_enrolled', 'sum'),
    total_days_absent=('days_absent', 'sum'),
    total_abs_excused=('days_excused', 'sum'),
    total_abs_unexcused=('days_unexcused', 'sum'),
    total_abs_suspended=('days_suspended', 'sum')
).reset_index()

Leaders would need to see attendance rate and absentee rate at a glance.

In [38]:
# weighted school-level attendance rate
school_summary['attendance_rate'] = (
    (school_summary['total_days_enrolled'] - school_summary['total_days_absent'])
    / school_summary['total_days_enrolled']
)

school_summary['absence_rate'] = 1 - school_summary['attendance_rate']

I believe it would also be useful to see the percentages of absences by category.

In [39]:
school_summary['pct_abs_excused'] = school_summary['total_abs_excused'] / school_summary['total_days_absent']
school_summary['pct_abs_unexcused'] = school_summary['total_abs_unexcused'] / school_summary['total_days_absent']
school_summary['pct_abs_suspended'] = school_summary['total_abs_suspended'] / school_summary['total_days_absent']

Finally, it would be good to see what percentage of students are chronically absent, at-risk, and satisfactory.

In [40]:
# count students by attendance_status per school
status_counts = students.groupby(['school', 'attendance_status'])['student_id'].count().unstack(fill_value=0).reset_index()

# merge with school_summary
school_summary = school_summary.merge(status_counts, on='school', how='left')

# calculate percentages
school_summary['pct_chronic'] = (school_summary.get('Chronic', 0) / school_summary['total_students'])
school_summary['pct_at_risk'] = (school_summary.get('At-Risk', 0) / school_summary['total_students'])
school_summary['pct_satisfactory'] = (school_summary.get('Satisfactory', 0) / school_summary['total_students'])

# drop raw columns
school_summary = school_summary.drop(columns=['Chronic', 'At-Risk', 'Satisfactory'])

In [41]:
school_summary

,school,total_students,total_days_enrolled,total_days_absent,total_abs_excused,total_abs_unexcused,total_abs_suspended,attendance_rate,absence_rate,pct_abs_excused,pct_abs_unexcused,pct_abs_suspended,pct_chronic,pct_at_risk,pct_satisfactory
0,Littlebrook Elementary,273,43889,3712,748,2542,422,0.915423,0.084577,0.201509,0.684806,0.113685,0.366300,0.311355,0.322344
1,North Star Middle,275,45206,3764,2677,562,525,0.916737,0.083263,0.711211,0.149309,0.139479,0.305455,0.316364,0.378182
2,Stevenson High,752,121839,12128,2791,7681,1656,0.900459,0.099541,0.230129,0.633328,0.136544,0.380319,0.281915,0.337766


In [42]:
school_summary.describe()

,total_students,total_days_enrolled,total_days_absent,total_abs_excused,total_abs_unexcused,total_abs_suspended,attendance_rate,absence_rate,pct_abs_excused,pct_abs_unexcused,pct_abs_suspended,pct_chronic,pct_at_risk,pct_satisfactory
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,433.333333,70311.333333,6534.666667,2072.000000,3595.000000,867.666667,0.910873,0.089127,0.380950,0.489148,0.129903,0.350691,0.303211,0.346097
std,275.975240,44629.126659,4844.038535,1148.033536,3674.458736,684.656361,0.009043,0.009043,0.286373,0.295432,0.014121,0.039798,0.018612,0.028836
min,273.000000,43889.000000,3712.000000,748.000000,562.000000,422.000000,0.900459,0.083263,0.201509,0.149309,0.113685,0.305455,0.281915,0.322344
25%,274.000000,44547.500000,3738.000000,1712.500000,1552.000000,473.500000,0.907941,0.083920,0.215819,0.391319,0.125114,0.335877,0.296635,0.330055
50%,275.000000,45206.000000,3764.000000,2677.000000,2542.000000,525.000000,0.915423,0.084577,0.230129,0.633328,0.136544,0.366300,0.311355,0.337766
75%,513.500000,83522.500000,7946.000000,2734.000000,5111.500000,1090.500000,0.916080,0.092059,0.470670,0.659067,0.138011,0.373310,0.313859,0.357974
max,752.000000,121839.000000,12128.000000,2791.000000,7681.000000,1656.000000,0.916737,0.099541,0.711211,0.684806,0.139479,0.380319,0.316364,0.378182


In [43]:
# sanity check
school_summary['pct_at_risk'][school_summary['school'] == 'Littlebrook Elementary'] + school_summary['pct_chronic'][school_summary['school'] == 'Littlebrook Elementary'] + school_summary['pct_satisfactory'][school_summary['school'] == 'Littlebrook Elementary']

0    1.0
dtype: float64

In [44]:
# sanity check
school_summary['pct_abs_excused'][school_summary['school'] == 'Littlebrook Elementary'] + school_summary['pct_abs_unexcused'][school_summary['school'] == 'Littlebrook Elementary'] + school_summary['pct_abs_suspended'][school_summary['school'] == 'Littlebrook Elementary']

0    1.0
dtype: float64

The school level view is ready for export!

---

## Export the Data

Export the student and school level data to Excel, with each DataFrame as its own tab.

In [45]:
# create the processed path
processed_dir = data_dir.parent / 'processed'
# print(processed_dir)

# path to save excel
output_path = processed_dir / 'MSCS_Attendance_Summary.xlsx'
# print(output_path)

In [46]:
# export the student and school level data into separate tabs in one excel file
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # student level tab
    students.to_excel(writer, sheet_name='Student_Detail', index=False)
    
    # school level tab
    school_summary.to_excel(writer, sheet_name='School_Summary', index=False)

print(f"Excel file saved with multiple tabs at {output_path}")

Excel file saved with multiple tabs at /Users/ajgafford/Documents/Projects/Python/MSCS/data/processed/MSCS_Attendance_Summary.xlsx
